# Supermarket Sales Analytics - Data Cleaning & Preparation

This notebook focuses on cleaning the raw dataset and creating additional business features for analysis.

## Objectives
- Clean and standardize data
- Create time-based features
- Add business metrics (profit, margin, customer insights)
- Prepare dataset for analysis and dashboarding

In [30]:
import pandas as pd
import numpy as np

In [31]:
# -------------------------------
# STEP 1: Load the raw dataset
# -------------------------------

# Read the CSV file from the raw data folder
# encoding='latin1' is used to handle special characters safely
sales_data = pd.read_csv("../data/raw/supermarket.csv", encoding="latin1")

# Check dataset size (rows, columns)
print("Dataset Shape:", sales_data.shape)

# Preview first few rows to understand structure
sales_data.head()


# -------------------------------
# STEP 2: Standardize column names
# -------------------------------

# Clean column names for consistency:
# - remove extra spaces
# - convert to lowercase
# - replace spaces and hyphens with underscores
sales_data.columns = (
    sales_data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

# View cleaned column names
sales_data.columns


# -------------------------------
# STEP 3: Convert date columns
# -------------------------------

# Convert 'order_date' to datetime format
# dayfirst=True ensures correct parsing for format like 15/04/2018
# errors='coerce' will convert invalid values to NaT instead of crashing
sales_data["order_date"] = pd.to_datetime(
    sales_data["order_date"],
    dayfirst=True,
    errors="coerce"
)

# Convert 'ship_date' to datetime format
sales_data["ship_date"] = pd.to_datetime(
    sales_data["ship_date"],
    dayfirst=True,
    errors="coerce"
)


# -------------------------------
# STEP 4: Validate date conversion
# -------------------------------

# Check if any values failed to convert (NaT = missing datetime)
print(sales_data[["order_date", "ship_date"]].isnull().sum())

Dataset Shape: (9800, 18)
order_date    0
ship_date     0
dtype: int64


In [32]:
# -------------------------------
# STEP 5: Create time-based features
# -------------------------------

# Extract year from order date
sales_data["order_year"] = sales_data["order_date"].dt.year

# Extract month number from order date
sales_data["order_month"] = sales_data["order_date"].dt.month

# Extract month name for easier reporting and charts
sales_data["order_month_name"] = sales_data["order_date"].dt.month_name()

# Extract day name to analyse weekday sales patterns
sales_data["order_day_name"] = sales_data["order_date"].dt.day_name()

# Extract quarter for higher-level time analysis
sales_data["order_quarter"] = sales_data["order_date"].dt.quarter

# Create weekend flag: True for Saturday/Sunday, False otherwise
sales_data["is_weekend"] = sales_data["order_date"].dt.weekday >= 5

In [33]:
# Preview the new time-based columns
sales_data[[
    "order_date",
    "order_year",
    "order_month",
    "order_month_name",
    "order_day_name",
    "order_quarter",
    "is_weekend"
]].head()

,order_date,order_year,order_month,order_month_name,order_day_name,order_quarter,is_weekend
0,2017-11-08,2017,11,November,Wednesday,4,False
1,2017-11-08,2017,11,November,Wednesday,4,False
2,2017-06-12,2017,6,June,Monday,2,False
3,2016-10-11,2016,10,October,Tuesday,4,False
4,2016-10-11,2016,10,October,Tuesday,4,False


In [34]:
# -------------------------------
# Create estimated cost columns
# -------------------------------

import numpy as np

# Keep generated values consistent each run
np.random.seed(42)

# Create quantity if it does not exist
if "quantity" not in sales_data.columns:
    sales_data["quantity"] = np.random.randint(1, 8, size=len(sales_data))

# Create discount if it does not exist
if "discount" not in sales_data.columns:
    sales_data["discount"] = np.random.choice(
        [0.00, 0.05, 0.10, 0.15, 0.20, 0.30],
        size=len(sales_data),
        p=[0.45, 0.15, 0.15, 0.10, 0.10, 0.05]
    )

# Define cost ratio ranges by category
category_cost_ranges = {
    "Furniture": (0.55, 0.75),
    "Office Supplies": (0.40, 0.65),
    "Technology": (0.65, 0.85)
}

# Function to estimate a cost ratio for each category
def estimate_cost_ratio(product_category):
    lower_bound, upper_bound = category_cost_ranges.get(product_category, (0.50, 0.70))
    return np.random.uniform(lower_bound, upper_bound)

# Create estimated cost ratio
sales_data["estimated_cost_ratio"] = sales_data["category"].apply(estimate_cost_ratio)

# Create estimated total cost
sales_data["estimated_cost"] = sales_data["sales"] * sales_data["estimated_cost_ratio"]

In [35]:
# -------------------------------
# STEP 6: Create pricing and profitability features
# -------------------------------

import numpy as np

# Set random seed so generated values stay the same every time the notebook runs
np.random.seed(42)

# Create quantity column if it does not exist in the raw dataset
# This helps us calculate unit-level pricing and order behaviour later
if "quantity" not in sales_data.columns:
    sales_data["quantity"] = np.random.randint(1, 8, size=len(sales_data))

# Create discount column if it does not exist
# Values are chosen from common retail discount levels
if "discount" not in sales_data.columns:
    sales_data["discount"] = np.random.choice(
        [0.00, 0.05, 0.10, 0.15, 0.20, 0.30],
        size=len(sales_data),
        p=[0.45, 0.15, 0.15, 0.10, 0.10, 0.05]
    )

# Recalculate sales after discount if you want a more commercial scenario
# Here we treat the original sales column as final sales value and estimate discount value separately
sales_data["estimated_discount_value"] = sales_data["sales"] * sales_data["discount"]

# Estimate unit selling price using sales and quantity
sales_data["estimated_unit_price"] = sales_data["sales"] / sales_data["quantity"]

# Estimate unit cost using the already-created estimated_cost column
sales_data["estimated_unit_cost"] = sales_data["estimated_cost"] / sales_data["quantity"]

# Create estimated profit using sales minus estimated cost
sales_data["estimated_profit"] = sales_data["sales"] - sales_data["estimated_cost"]

# Create estimated profit margin
sales_data["estimated_profit_margin"] = np.where(
    sales_data["sales"] > 0,
    sales_data["estimated_profit"] / sales_data["sales"],
    0
)

In [36]:
# Preview new pricing-related columns
sales_data[[
    "sales",
    "quantity",
    "discount",
    "estimated_discount_value",
    "estimated_cost",
    "estimated_unit_price",
    "estimated_unit_cost",
    "estimated_profit",
    "estimated_profit_margin"
]].head()

sales_data.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'order_year',
 'order_month',
 'order_month_name',
 'order_day_name',
 'order_quarter',
 'is_weekend',
 'quantity',
 'discount',
 'estimated_cost_ratio',
 'estimated_cost',
 'estimated_discount_value',
 'estimated_unit_price',
 'estimated_unit_cost',
 'estimated_profit',
 'estimated_profit_margin']

In [37]:
# -------------------------------
# STEP 7: Customer & Order-Level Insights
# -------------------------------

# -------------------------------
# Customer-level metrics
# -------------------------------

# Total revenue generated by each customer
sales_data["customer_total_revenue"] = (
    sales_data.groupby("customer_id")["sales"].transform("sum")
)

# Total number of unique orders per customer
sales_data["customer_order_count"] = (
    sales_data.groupby("customer_id")["order_id"].transform("nunique")
)

# Average order value per customer
sales_data["customer_avg_order_value"] = (
    sales_data["customer_total_revenue"] / sales_data["customer_order_count"]
)


# -------------------------------
# Customer segmentation
# -------------------------------

# Segment customers into Low, Medium, High value groups
sales_data["customer_value_segment"] = pd.qcut(
    sales_data["customer_total_revenue"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)


# -------------------------------
# Order-level metrics
# -------------------------------

# Total sales per order
sales_data["order_total_sales"] = (
    sales_data.groupby("order_id")["sales"].transform("sum")
)

# Total quantity per order
sales_data["order_total_quantity"] = (
    sales_data.groupby("order_id")["quantity"].transform("sum")
)

# Total profit per order (using estimated profit)
sales_data["order_total_profit"] = (
    sales_data.groupby("order_id")["estimated_profit"].transform("sum")
)


# -------------------------------
# Bulk order flag
# -------------------------------

# Identify large orders (useful for business insights)
sales_data["bulk_order_flag"] = np.where(
    sales_data["order_total_quantity"] >= 10,
    1,
    0
)

In [38]:
# Preview customer and order metrics
sales_data[[
    "customer_id",
    "customer_total_revenue",
    "customer_order_count",
    "customer_avg_order_value",
    "customer_value_segment",
    "order_id",
    "order_total_sales",
    "order_total_quantity",
    "order_total_profit",
    "bulk_order_flag"
]].head()

,customer_id,customer_total_revenue,customer_order_count,customer_avg_order_value,customer_value_segment,order_id,order_total_sales,order_total_quantity,order_total_profit,bulk_order_flag
0,CG-12520,1148.7800,3,382.926667,Low Value,CA-2017-152156,993.9000,11,301.945247,1
1,CG-12520,1148.7800,3,382.926667,Low Value,CA-2017-152156,993.9000,11,301.945247,1
2,DV-13045,1119.4830,5,223.896600,Low Value,CA-2017-138688,14.6200,5,7.926099,0
3,SO-20335,2602.5755,6,433.762583,Medium Value,US-2016-108966,979.9455,10,393.508521,1
4,SO-20335,2602.5755,6,433.762583,Medium Value,US-2016-108966,979.9455,10,393.508521,1


In [39]:
# -------------------------------
# STEP 8: Inventory Simulation & Stock Analysis
# -------------------------------

# Create a product-level inventory table (one row per product)
product_inventory = (
    sales_data[["product_id", "category", "sub_category"]]
    .drop_duplicates()
    .copy()
)

# Set seed for consistent results
np.random.seed(42)

# Simulate stock quantity (current stock available)
product_inventory["stock_quantity"] = np.random.randint(20, 250, size=len(product_inventory))

# Simulate reorder level (threshold to trigger restocking)
product_inventory["reorder_level"] = np.random.randint(15, 60, size=len(product_inventory))


# -------------------------------
# Merge inventory data back into main dataset
# -------------------------------

sales_data = sales_data.merge(
    product_inventory[["product_id", "stock_quantity", "reorder_level"]],
    on="product_id",
    how="left"
)


# -------------------------------
# Create stock status indicator
# -------------------------------

# Identify whether a product needs restocking
sales_data["stock_status"] = np.where(
    sales_data["stock_quantity"] <= sales_data["reorder_level"],
    "Reorder Needed",
    "Stock OK"
)


# -------------------------------
# Create inventory risk flag
# -------------------------------

# Identify products that are selling well but have low stock
sales_threshold = sales_data["sales"].quantile(0.75)

sales_data["high_demand_low_stock_flag"] = np.where(
    (sales_data["sales"] >= sales_threshold) &
    (sales_data["stock_quantity"] <= sales_data["reorder_level"]),
    1,
    0
)

In [40]:
# Preview inventory-related columns
sales_data[[
    "product_id",
    "category",
    "stock_quantity",
    "reorder_level",
    "stock_status",
    "high_demand_low_stock_flag"
]].head()

,product_id,category,stock_quantity,reorder_level,stock_status,high_demand_low_stock_flag
0,FUR-BO-10001798,Furniture,122,18,Stock OK,0
1,FUR-CH-10000454,Furniture,199,43,Stock OK,0
2,OFF-LA-10000240,Office Supplies,112,36,Stock OK,0
3,FUR-TA-10000577,Furniture,34,39,Reorder Needed,1
4,OFF-ST-10000760,Office Supplies,126,27,Stock OK,0


In [41]:
# -------------------------------
# STEP 9: Business Risk Flags & Insight Indicators
# -------------------------------

# -------------------------------
# Loss-making transactions
# -------------------------------

# Flag transactions where estimated profit is negative
sales_data["loss_making_flag"] = np.where(
    sales_data["estimated_profit"] < 0,
    1,
    0
)


# -------------------------------
# High discount risk
# -------------------------------

# Flag transactions with high discount (>= 30%)
sales_data["high_discount_flag"] = np.where(
    sales_data["discount"] >= 0.3,
    1,
    0
)


# -------------------------------
# High sales but low profit (KEY BUSINESS INSIGHT)
# -------------------------------

# Define thresholds using percentiles
high_sales_threshold = sales_data["sales"].quantile(0.75)
low_profit_threshold = sales_data["estimated_profit"].quantile(0.25)

# Flag transactions with strong sales but weak profitability
sales_data["high_sales_low_profit_flag"] = np.where(
    (sales_data["sales"] >= high_sales_threshold) &
    (sales_data["estimated_profit"] <= low_profit_threshold),
    1,
    0
)


# -------------------------------
# Shipping performance analysis
# -------------------------------

# Calculate delivery time in days
sales_data["shipping_days"] = (
    sales_data["ship_date"] - sales_data["order_date"]
).dt.days

# Categorise shipping speed
sales_data["shipping_speed"] = pd.cut(
    sales_data["shipping_days"],
    bins=[-1, 2, 5, 100],
    labels=["Fast", "Standard", "Slow"]
)


# -------------------------------
# Regional performance classification
# -------------------------------

# Calculate total sales per region
region_sales_summary = sales_data.groupby("region")["sales"].sum()

# Calculate average regional sales
average_region_sales = region_sales_summary.mean()

# Map regions to performance category
sales_data["region_performance"] = sales_data["region"].map(
    lambda region: "High Performing"
    if region_sales_summary[region] >= average_region_sales
    else "Low Performing"
)

In [42]:
# Preview risk and insight columns
sales_data[[
    "sales",
    "estimated_profit",
    "loss_making_flag",
    "discount",
    "high_discount_flag",
    "high_sales_low_profit_flag",
    "shipping_days",
    "shipping_speed",
    "region",
    "region_performance"
]].head()

,sales,estimated_profit,loss_making_flag,discount,high_discount_flag,high_sales_low_profit_flag,shipping_days,shipping_speed,region,region_performance
0,261.9600,68.538100,0,0.05,0,0,3,Standard,South,Low Performing
1,731.9400,233.407147,0,0.00,0,0,3,Standard,South,Low Performing
2,14.6200,7.926099,0,0.00,0,0,4,Standard,West,High Performing
3,957.5775,382.696908,0,0.05,0,0,7,Slow,South,Low Performing
4,22.3680,10.811613,0,0.20,0,0,7,Slow,South,Low Performing


In [43]:
# -------------------------------
# Save final enhanced dataset
# -------------------------------

sales_data.to_csv("../data/processed/enhanced_supermarket.csv", index=False)

print("Final enhanced dataset saved successfully.")

Final enhanced dataset saved successfully.
